*0.3 Classical NLP*

# Tokenization: WordPiece

**The situation.** Your search re-ranker is a BERT model. A colleague asks why its tokens have `##` in them while GPT's do not, and whether "un" + "##believable" and "unbelievable" are the same to the model. They are not — and the answer matters for how you clean queries before ranking.

**WordPiece.** BERT's tokenizer. Like BPE it builds pieces by merging, but it picks the merge that most improves the *likelihood* of the training text rather than the most frequent pair. At tokenization time it is greedy: take the longest piece in the vocabulary that matches the start of the word, then continue. Pieces that continue a word are written with `##`.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
for text in ["unbelievable", "un believable", "chargebacks", "The invoice was unpaid."]:
    encoded = tokenizer(text)
    print(f"{text!r:<26} → {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")
assert tokenizer.tokenize("unbelievable") != tokenizer.tokenize("un believable")

'unbelievable'             → ['[CLS]', 'unbelievable', '[SEP]']
'un believable'            → ['[CLS]', 'un', 'bel', '##ie', '##vable', '[SEP]']
'chargebacks'              → ['[CLS]', 'charge', '##backs', '[SEP]']
'The invoice was unpaid.'  → ['[CLS]', 'the', 'in', '##vo', '##ice', 'was', 'unpaid', '.', '[SEP]']


**Reading the output.** "unbelievable" → `un` + `##bel` + … ; "un believable" → `un` + `believable` — different pieces, so different input to the model. The full sentence shows the `[CLS]` and `[SEP]` markers BERT expects around every input, added by the tokenizer.

**Longest match first, by hand.** The same greedy rule, spelled out for one word.

In [3]:
vocabulary = tokenizer.get_vocab()


def wordpiece(word: str) -> list[str]:
    pieces = []
    start = 0
    while start < len(word):
        end = len(word)
        while end > start:  # try the longest candidate first
            candidate = word[start:end] if start == 0 else "##" + word[start:end]
            if candidate in vocabulary:
                pieces.append(candidate)
                break
            end -= 1
        if end == start:
            return ["[UNK]"]
        start = end
    return pieces


print("by hand:  ", wordpiece("chargebacks"))
print("tokenizer:", tokenizer.tokenize("chargebacks"))
assert wordpiece("chargebacks") == tokenizer.tokenize("chargebacks")

by hand:   ['charge', '##backs']
tokenizer: ['charge', '##backs']


**The rule to remember.** WordPiece = BERT family. `##` means "continues the word". Longest-known-piece-first decides the split, so spacing and casing change the tokens.

| Use it when | Don't when | Instead use |
|---|---|---|
| any BERT-style encoder (re-rankers, classifiers, embedding models) | GPT-style models | BPE / tiktoken |

**Watch out**
- `bert-base-uncased` lowercases everything; the cased variant does not. Match the model you serve.
- Every BERT input has a 512-token limit including `[CLS]`/`[SEP]`; count with the tokenizer, not with words.
- A word with no matching piece becomes `[UNK]` — possible for unusual scripts; check for it in logs.